In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib
import pickle
from sklearn.model_selection import train_test_split

```
Загрузка датасета
```

In [3]:
df_load = pd.read_csv('data/drom_archive_cleaned_2018-2025full.csv')
df = df_load

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4395695 entries, 0 to 4395694
Data columns (total 14 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Год              float64
 1   Цена             float64
 2   Объем двигателя  float64
 3   Тип двигателя    object 
 4   Мощность         float64
 5   Коробка передач  object 
 6   Привод           object 
 7   Пробег           float64
 8   Поколение        float64
 9   Тип кузова       object 
 10  Метка            object 
 11  Город            object 
 12  Год объявления   int64  
 13  Возраст авто     float64
dtypes: float64(7), int64(1), object(6)
memory usage: 469.5+ MB


## Линейная регрессия

```
Преобразование категориальных признаков
```

In [5]:
categorical = ['Тип двигателя', 'Коробка передач', 'Привод', 'Поколение', 'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Возраст авто']

In [6]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical), 
    ('num', 'passthrough', numerical)], remainder='drop')

In [7]:
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])

```
Разделение данных на целевую переменную и признаки
```

```
Разделение на обучающую и тестовую выборки
```

In [8]:
y = df['Цена']
X = df.drop('Цена', axis=1)

In [9]:
df['price_stratum'] = pd.qcut(df['Цена'], q=10, labels=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df['price_stratum'])

```
Обучение и сохранение модели
```

```
Предсказание на тестовой выборке
```

In [10]:
model.fit(X_train, y_train)
joblib.dump(model, "models/lr_model.pkl")

['models/lr_model.pkl']

In [11]:
y_pred = model.predict(X_test)

```
Оценка модели
```

```
Вывод результатов
```

In [12]:
lr_mse = mean_squared_error(y_test, y_pred)
lr_rmse = np.sqrt(lr_mse)
lr_mae = mean_absolute_error(y_test, y_pred)
lr_r2 = r2_score(y_test, y_pred)
lr_mape = mean_absolute_percentage_error(y_test, y_pred)

In [13]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 
                     'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)', 
                     'Средняя абсолютная процентная ошибка (MAPE)'],
    'Результаты': [lr_mse, lr_rmse, lr_mae, lr_r2, lr_mape]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),98_330_108_155.26
1,Среднеквадратическая ошибка (RMSE),313_576.32
2,Средняя абсолютная ошибка (MAE),230_266.28
3,Коэффицент детерминации (R^2),0.67
4,Средняя абсолютная процентная ошибка (MAPE),0.24


```
Сохранение метрик в отдельный файл
```

In [14]:
metrics = {
    "model_name": "Linear Regression",
    "MSE": lr_mse,
    "RMSE": lr_rmse,
    "MAE": lr_mae,
    "R2": lr_r2,
    "MAPE": lr_mape
}

In [15]:
with open("metrics/lr_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)